In [ ]:
!pip install -q datasets transformers huggingface_hub

In [ ]:
import os
import json
import random
import tempfile
from tqdm.auto import tqdm

from datasets import load_dataset
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
login(HF_TOKEN)

HF_USER = "yasnansh"

SOURCE_CONFIGS = {
    "scientific": "common-pile/peS2o_filtered",
    "knowledge":  "common-pile/wikimedia_filtered",
    "web_text":   "common-pile/cccc_filtered",
    "code":       "common-pile/stackv2_edu_filtered",
}

TARGET_PERCENTS = {
    "scientific": 0.3,
    "knowledge":  0.3,
    "web_text":   0.2,
    "code":       0.2,
}

TOTAL_QUERIES = 80_000
TOTAL_DOCS = 3_600_000
BASE_WORDS = 128

SHUFFLE_BUFFER = 50_000            # streaming shuffle buffer for source dataset (per-category)
WRITE_BATCH_SIZE = 1_000           # flush JSONL lines every batch to reduce disk IO overhead
MERGE_SHUFFLE_BUFFER = 200_000     # buffer size used when producing a globally-shuffled merged file

# Text column name in the source datasets (change if different)
TEXT_COLUMN = "text"

RANDOM_SEED = 23
random.seed(RANDOM_SEED)


def base_unit_generator(streaming_dataset, base_words=BASE_WORDS):
    """
    Given a streaming HF dataset (each element with a TEXT_COLUMN), yield successive
    non-overlapping base-unit chunks of `base_words` words as strings.
    Leftover words at the end are discarded.
    """
    for example in streaming_dataset:
        text = example.get(TEXT_COLUMN, None)
        if not text:
            continue
        # split on whitespace (simple, fast)
        words = text.split()
        if not words:
            continue
        if len(words) >= 2 * base_words:
            start = random.randint(0, len(words) - 2 * base_words)
            yield words[start:start+2 * base_words]
    return

def stream_and_write_category(
    category,
    hf_source_path,
    n_queries,
    n_docs,
    hf_user=HF_USER,
    tmp_dir=None,
    shuffle_buffer=SHUFFLE_BUFFER,
    base_words=BASE_WORDS,
):
    """
    Process one category:
      - load source streaming + shuffle
      - create base_unit stream (128 words)
      - allocate first 2*n_queries base_units to make queries (pairs),
        and the next n_docs base_units to make docs.
      - write two JSONL files: {category}_queries.jsonl and {category}_docs.jsonl
      - push each to HF under hf_user/{category}_queries_train  (and docs)
    Returns repo ids for queries and docs.
    """
    if tmp_dir is None:
        tmp_dir = tempfile.mkdtemp(prefix=f"{category}_tmp_")
    os.makedirs(tmp_dir, exist_ok=True)
    q_path = os.path.join(tmp_dir, f"{category}_queries.jsonl")
    d_path = os.path.join(tmp_dir, f"{category}_docs.jsonl")

    needed_base_units = n_queries + n_docs
    print(f"[{category}] Need base units: {needed_base_units} (queries: {n_queries}, docs: {n_docs})")

    # Streaming load and shuffle
    print(f"[{category}] Loading streaming dataset: {hf_source_path} (shuffle buffer={shuffle_buffer})")
    ds_stream = load_dataset(hf_source_path, split="train", streaming=True)
    try:
        ds_stream = ds_stream.shuffle(buffer_size=shuffle_buffer, seed=RANDOM_SEED)
    except Exception:
        # Some datasets/versions may not support streaming shuffle; fall back to no-shuffle (still streaming)
        print(f"[{category}] Warning: streaming.shuffle() not supported, proceeding without pre-shuffle.")
        pass

    base_gen = base_unit_generator(ds_stream, base_words=base_words)

    # Open files and stream write
    q_file = open(q_path, "w", encoding="utf-8")
    d_file = open(d_path, "w", encoding="utf-8")

    written_q = 0
    written_d = 0
    i = 0
    batch_q = []
    batch_d = []

    pbar = tqdm(total=needed_base_units, desc=f"[{category}] base_units", unit="unit")
    try:
        while i < needed_base_units:
            try:
                unit = next(base_gen)
            except StopIteration:
                print(f"[{category}] Reached end of stream early after {i} base units.")
                break

            if i < n_queries:
                # queries: pair two base units -> query (first base unit) + answer (second base unit)
                q_obj = {"query": " ".join(unit[:base_words]), "answer": " ".join(unit[base_words:])}
                batch_q.append(json.dumps(q_obj, ensure_ascii=False))
                written_q += 1
                if len(batch_q) >= WRITE_BATCH_SIZE:
                    q_file.write("\n".join(batch_q) + "\n")
                    batch_q = []
            else:
                # docs: single base unit per doc
                if random.random() < 0.5:
                    d_obj = {"text": " ".join(unit[:base_words])}
                else:
                    d_obj = {"text": " ".join(unit[base_words:])}
                    
                batch_d.append(json.dumps(d_obj, ensure_ascii=False))
                written_d += 1
                if len(batch_d) >= WRITE_BATCH_SIZE:
                    d_file.write("\n".join(batch_d) + "\n")
                    batch_d = []

            i += 1
            pbar.update(1)
    finally:
        # flush remaining
        if batch_q:
            q_file.write("\n".join(batch_q) + "\n")
            batch_q = []
        if batch_d:
            d_file.write("\n".join(batch_d) + "\n")
            batch_d = []
        q_file.close()
        d_file.close()
        pbar.close()

    print(f"[{category}] Written queries: {written_q}, docs: {written_d}")
    # Now load each JSONL as HF Dataset and push
    from datasets import load_dataset as load_ds_local

    # push queries
    q_ds = load_ds_local("json", data_files=q_path, split="train")
    q_repo = f"{hf_user}/{category}_queries_train"
    print(f"[{category}] Pushing queries to hub: {q_repo}")
    q_ds.push_to_hub(q_repo, private=False)

    # push docs
    d_ds = load_ds_local("json", data_files=d_path, split="train")
    d_repo = f"{hf_user}/{category}_docs_train"
    print(f"[{category}] Pushing docs to hub: {d_repo}")
    d_ds.push_to_hub(d_repo, private=False)

    # cleanup local files to save disk
    try:
        os.remove(q_path)
        os.remove(d_path)
        os.rmdir(tmp_dir)
    except Exception:
        pass

    return q_repo, d_repo

def build_all_categories(
    source_configs=SOURCE_CONFIGS,
    target_percents=TARGET_PERCENTS,
    total_queries=TOTAL_QUERIES,
    total_docs=TOTAL_DOCS,
):
    query_repos = {}
    doc_repos = {}
    for category, hf_path in source_configs.items():
        pct = target_percents.get(category, 0.0)
        n_q = int(total_queries * pct)
        n_d = int(total_docs * pct)
        if n_q == 0 and n_d == 0:
            print(f"Skipping {category} (zero target).")
            continue
        q_repo, d_repo = stream_and_write_category(
            category=category,
            hf_source_path=hf_path,
            n_queries=n_q,
            n_docs=n_d
        )
        query_repos[category] = q_repo
        doc_repos[category] = d_repo
    return query_repos, doc_repos

# ----------- Optional: Merge per-category HF datasets into single global dataset (streaming + block-shuffle) -----------
def merge_and_shuffle_hf_datasets_to_jsonl(repo_list, out_path, buffer_size=MERGE_SHUFFLE_BUFFER):
    """
    repo_list: list of HF repo ids (e.g., ['yasnansh/scientific_docs_train', ...])
    out_path: path to write final JSONL
    This does a block-shuffle: it fills an in-memory buffer up to buffer_size,
    then random.shuffle(buffer) and flush. This approximates global shuffle with fixed memory.
    """
    out_file = open(out_path, "w", encoding="utf-8")
    buffer = []
    total_in = 0

    # Make a combined generator that yields examples from each repo streaming
    def combined_generator(repos):
        for repo in repos:
            ds = load_dataset(repo, split="train", streaming=True)
            for ex in ds:
                yield ex

    print(f"Merging {len(repo_list)} repos into {out_path} (buffer_size={buffer_size})")
    gen = combined_generator(repo_list)
    pbar = tqdm(desc="merged", unit="examples")
    try:
        for ex in gen:
            buffer.append(ex)
            total_in += 1
            pbar.update(1)
            if len(buffer) >= buffer_size:
                random.shuffle(buffer)
                # write and clear
                for r in buffer:
                    out_file.write(json.dumps(r, ensure_ascii=False) + "\n")
                buffer = []
        # flush remainder
        if buffer:
            random.shuffle(buffer)
            for r in buffer:
                out_file.write(json.dumps(r, ensure_ascii=False) + "\n")
            buffer = []
    finally:
        out_file.close()
        pbar.close()
    print(f"Merge complete. Total examples written: {total_in}")

# ----------- MAIN (execute pipeline) -----------
if __name__ == "__main__":
    # 1) Process and push per-category splits
    print("Starting per-category processing...")
    query_repos, doc_repos = build_all_categories()

    print("\nPer-category push completed. Query repos:")
    for k, v in query_repos.items():
        print(f"  {k}: {v}")
    print("Doc repos:")
    for k, v in doc_repos.items():
        print(f"  {k}: {v}")

    # 2) (OPTIONAL) Merge all query repos into a single file (locally) and same for docs.
    #    You may want final shuffling across categories; do this only if you have enough disk.
    #    Uncomment and run if you want merged JSONL outputs.
    #
    # MERGE_QUERIES_OUT = "/kaggle/working/all_queries_merged.jsonl"
    # MERGE_DOCS_OUT = "/kaggle/working/all_docs_merged.jsonl"
    #
    # print("\nMerging query repos into a global shuffled JSONL (this may take time)...")
    # merge_and_shuffle_hf_datasets_to_jsonl(list(query_repos.values()), MERGE_QUERIES_OUT)
    # print("\nMerging doc repos into a global shuffled JSONL (this may take time)...")
    # merge_and_shuffle_hf_datasets_to_jsonl(list(doc_repos.values()), MERGE_DOCS_OUT)
    #
    # After merging you can load the merged JSONL with `load_dataset("json", data_files=...)` and push as a final hub dataset.